# Train Seq2SeqSimplifier trên Colab

Notebook này chỉ lo phần môi trường (clone code, gắn dữ liệu/model từ Drive, cài thư viện) —
logic train thật sự nằm trong repo (`src/simplification/seq2seq_simplifier.py`,
`scripts/simplification/train_seq2seq_simplifier.py`), đồng bộ qua git.

**Trước khi chạy:**
1. Đổi `DRIVE_ROOT` bên dưới nếu bạn để dữ liệu ở thư mục Drive khác.
2. Đã upload sẵn `data/dataset/simplificated_spo_sentence.csv` vào `DRIVE_ROOT/data/dataset/simplificated_spo_sentence.csv` trên Drive.
3. Chọn Runtime > Change runtime type > GPU trước khi chạy (nếu có GPU free trên Colab).


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Clone / pull code từ git

Repo public, không cần token. Nếu đã clone từ lần trước, cell này sẽ `git pull` thay vì clone lại.

In [ ]:
REPO_URL = "https://github.com/thnghia-ctu/CausalGraph.git"
BRANCH = "v3"
REPO_DIR = "/content/CausalGraph"

import os

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR


Cloning into '/content/CausalGraph'...
remote: Enumerating objects: 896, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 896 (delta 81), reused 108 (delta 40), pack-reused 718 (from 1)
Receiving objects: 100% (896/896), 1.87 MiB | 3.08 MiB/s, done.
Resolving deltas: 100% (464/464), done.
/content/CausalGraph


## 3. Gắn `data/` và `models/` vào Drive

Hai thư mục này bị `.gitignore`, không nằm trong git — clone xong sẽ trống hoặc không tồn tại.
Symlink sang Drive để dữ liệu và checkpoint được giữ lại qua các session, không cần copy tay
mỗi lần mở lại Colab.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/CausalGraph"

import os

os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)

!rm -rf {REPO_DIR}/data {REPO_DIR}/models
!ln -s {DRIVE_ROOT}/data {REPO_DIR}/data
!ln -s {DRIVE_ROOT}/models {REPO_DIR}/models

!ls -la {REPO_DIR}/data/dataset/ 2>/dev/null || echo "Chưa có data/dataset/simplificated_spo_sentence.csv trên Drive — upload trước khi train."


total 6295
-rw------- 1 root root 2926373 Aug  9 15:15 causal_sentences.csv
-rw------- 1 root root   40352 Aug 11 08:08 gold_simplification_candidates.xlsx
-rw------- 1 root root 3478666 Aug 11 10:24 simplificated_spo_sentence.csv


## 4. Cài thư viện

In [ ]:
!pip install -q -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 21.1 MB/s eta 0:00:00
   ━

## 5. Đăng nhập Hugging Face Hub

Bắt buộc — cell train ở bước 6 giờ tự push model lên Hub ngay khi train xong
(dùng repo ID khai trong `configs/config.py`). Token tạo tại
https://huggingface.co/settings/tokens (quyền write).


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 6. Train

Checkpoint được lưu định kỳ vào `models/simplifier/` (= Drive, qua symlink ở bước 3).
Nếu Colab bị ngắt kết nối giữa chừng, chỉ cần chạy lại cell này — `Seq2SeqSimplifier.fit`
tự resume từ checkpoint gần nhất thay vì train lại từ đầu. Train xong sẽ tự
push model lên Hugging Face Hub (repo ID lấy từ configs/config.py).

In [ ]:
!python -m scripts.simplification.train_seq2seq_simplifier


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  9% 5691/61900 [22:52<3:52:12,  4.03it/s]

  9% 5692/61900 [22:53<3:46:00,  4.15it/s]

  9% 5693/61900 [22:53<3:41:45,  4.22it/s]

  9% 5694/61900 [22:53<3:44:51,  4.17it/s]

  9% 5695/61900 [22:53<3:43:26,  4.19it/s]

  9% 5696/61900 [22:54<3:45:23,  4.16it/s]

  9% 5697/61900 [22:54<3:44:22,  4.17it/s]

  9% 5698/61900 [22:54<3:54:01,  4.00it/s]

  9% 5699/61900 [22:54<4:06:58,  3.79it/s]

  9% 5700/61900 [22:55<4:36:55,  3.38it/s]

{'loss': '0.0161', 'grad_norm': '1.454', 'learning_rate': '2.724e-05', 'epoch': '9.208'}


  9% 5700/61900 [22:55<4:36:55,  3.38it/s]

  9% 5701/61900 [22:55<4:42:18,  3.32it/s]

  9% 5702/61900 [22:55<4:49:47,  3.23it/s]

  9% 5703/61900 [22:56<4:44:52,  3.29it/s]

  9% 5704/61900 [22:56<4:46:25,  3.27it/s]

  9% 5705/61900 [22:56<5:10:31,  3.02it/s]

  9% 5706/61900 [22:57<5:21:27,  2.91it/s]

  9% 5707/61900 [22:57<5:04:01,  3.08it/s]

  9% 5708/61900 [22:57<5:06:30,  3.06it/s]

  9% 5709/61900 